# DeepSC: validation results and experiment comparisons

Run this notebook from **pecSC / kd-exp** using the project Python environment.
It compares M1 (CE only), M2 (combined KD), feature-only and logit-only Students.
The Teacher transmitter remains fixed at epoch 38. Training seeds and paths are
editable below. All numerical results are read from CSVs/manifests, never hardcoded.

**Default:** analyze saved experiment records and export tables/figures. No GPU,
training, checkpoint modification or held-out test access is required.
Optional paired validation uses your local data/checkpoints and is disabled.

These are **teacher-forced validation** results. They do not establish
autoregressive BLEU, semantic similarity, inference latency or final test quality.
Keep the selected-checkpoint comparisons distinct from repeated channel trials.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from types import SimpleNamespace
import json, os, sys, subprocess, tempfile
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

try:
    from IPython.display import display, Markdown
except ImportError:  # Allows headless cell verification without a Jupyter kernel.
    display = print
    Markdown = str

def find_repo(start):
    for candidate in (start, *start.parents):
        if (candidate / 'train_multi_vocab_one_student.py').is_file() and (candidate / 'deepsc_reproduce').is_dir():
            return candidate
    raise FileNotFoundError('Open this notebook inside pecSC, or set REPO explicitly.')

REPO = find_repo(Path.cwd().resolve())  # Or Path('/home/prinako/dev/BPE/pecSC')
sys.path.insert(0, str(REPO))
from deepsc_reproduce.reproduce import read_run, control_signature, sha, git_state, diagnostics
from deepsc_reproduce.evaluate_paired_validation import paired_summary

TRAINING_SEEDS = [42, 43, 44, 45]
EPOCH_BUDGET = 10
if not TRAINING_SEEDS or len(set(TRAINING_SEEDS)) != len(TRAINING_SEEDS):
    raise ValueError('Choose a nonempty list of distinct training seeds.')
RUN_ROOT = REPO / 'results/article_kd/diagnostics'
PAIRED_ROOT = REPO / 'results/article_kd/paired_validation'
PAIRED_DIR = None  # Set an explicit full-evaluation directory, or auto-select below.
RUN_PAIRED_EVALUATION = False  # Explicit opt-in: requires local checkpoints/data + Torch.
CHANNEL_TRIAL_SEEDS = [1001, 1002, 1003, 1004, 1005]
RUN_DIAGNOSTICS = False  # Raw CSVs are large and intentionally excluded from Git.
DIAGNOSTIC_PATHS = []  # e.g. [RUN_ROOT / '...seed42' / 'diagnostics.csv']
SAVE_EXPORTS = True
EXPORT_ROOT = REPO / 'results/article_kd/notebook_reports'

plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 180, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.2})
LABELS = {'m1': 'M1: CE only', 'm2': 'M2: combined KD',
          'feature': 'CE + features', 'logit': 'CE + logit KD'}
COLORS = {'m1': '#2563eb', 'm2': '#d97706', 'feature': '#059669', 'logit': '#9333ea'}
TEMPLATES = {'m1': 'm1-rayleigh-adam-wd0-{epochs}epochs-seed{seed}',
             'm2': 'm2-rayleigh-adam-wd0-{epochs}epochs-seed{seed}',
             'feature': 'ablation-ce1-feature-wd0-{epochs}epochs-seed{seed}',
             'logit': 'ablation-ce1-logit-wd0-{epochs}epochs-seed{seed}'}
WEIGHTS = {'m1': (1., 0., 0.), 'm2': (0.6, 0.3, 0.1),
           'feature': (1., 0., 0.1), 'logit': (1., 0.3, 0.)}
figures, export_tables, source_identities = {}, {}, []
print('Repository:', REPO)
print('Python kernel:', sys.executable)
print('Git:', git_state(REPO))
print('Training seeds:', TRAINING_SEEDS)


## Optional: generate a new paired evaluation

Leave the switch **False** to use existing results. Enabling it runs eight saved
receivers against the same received tensor per validation batch, at Rayleigh 8 dB,
using five channel seeds. It does not train. The evaluator verifies local artifact
hashes and creates a fresh directory. This cell may take considerable time.


In [ ]:
if RUN_PAIRED_EVALUATION:
    if EPOCH_BUDGET != 10:
        raise ValueError('The current paired evaluator templates assume ten-epoch run names.')
    new_output = PAIRED_ROOT / ('notebook-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    command = [sys.executable, str(REPO / 'deepsc_reproduce/reproduce.py'), '--repo', str(REPO),
               'paired', '--run-root', str(RUN_ROOT), '--output-dir', str(new_output),
               '--seeds', *map(str, TRAINING_SEEDS),
               '--trial-seeds', *map(str, CHANNEL_TRIAL_SEEDS)]
    print('Running:', ' '.join(command), flush=True)
    subprocess.run(command, cwd=REPO, check=True)
    PAIRED_DIR = new_output
else:
    print('GPU reevaluation disabled. Reading saved records only.')


## 1. Load and verify experiment records

Best checkpoints are selected by minimum **val_ce**, not combined val_loss.
Manifest selection must agree with the CSV minimum. Comparisons require matched
architecture, dataset/Teacher hashes, optimizer, channel and other controls.
Only the configured canonical ten-epoch Student runs enter the main tables.
Missing arms are reported explicitly, never replaced by invented values.


In [ ]:
manifests, run_rows, epoch_rows, missing = {}, [], [], []
for seed in TRAINING_SEEDS:
    for variant, template in TEMPLATES.items():
        directory = RUN_ROOT / template.format(epochs=EPOCH_BUDGET, seed=seed)
        if not directory.is_dir():
            missing.append({'variant': variant, 'seed': seed, 'directory': str(directory)})
            continue
        manifest, rows, best = read_run(directory)
        if manifest['seed'] != seed:
            raise ValueError(f'Seed mismatch: {directory}')
        if tuple(manifest['loss'][k] for k in ('alpha', 'beta', 'gamma')) != WEIGHTS[variant]:
            raise ValueError(f'Wrong loss weights: {directory}')
        if manifest['optimizer']['weight_decay'] != 0:
            raise ValueError(f'Expected weight decay zero: {directory}')
        if len(rows) != EPOCH_BUDGET or manifest['epoch_budget'] != EPOCH_BUDGET:
            raise ValueError(f'Incomplete or different epoch budget: {directory}')
        manifests[(seed, variant)] = manifest
        run_rows.append({'seed': seed, 'variant': variant, 'model': LABELS[variant],
                         'best_epoch': int(best['epoch']), 'best_val_ce': float(best['val_ce']),
                         'final_train_ce': float(rows[-1]['ce']),
                         'minutes_per_epoch': np.mean([float(r['time']) for r in rows]) / 60,
                         'recorded_checkpoint_bytes': manifest['selected_checkpoint']['size_bytes'],
                         'checkpoint_sha256': manifest['selected_checkpoint']['sha256']})
        for row in rows:
            epoch_rows.append({'seed': seed, 'variant': variant,
                               **{k: float(v) for k, v in row.items() if k != 'epoch'},
                               'epoch': int(row['epoch'])})
        for filename in ('manifest.json', 'training.csv'):
            path = directory / filename
            source_identities.append({'path': str(path.relative_to(REPO)), 'sha256': sha(path)})

if not run_rows:
    raise FileNotFoundError(f'No configured experiment records under {RUN_ROOT}')
for (seed, variant), manifest in manifests.items():
    if (seed, 'm1') not in manifests:
        raise ValueError(f'Missing matched M1 baseline for seed {seed}')
    if control_signature(manifest) != control_signature(manifests[(seed, 'm1')]):
        raise ValueError(f'Unmatched experimental controls for seed {seed}, {variant}')

runs = pd.DataFrame(run_rows).sort_values(['seed', 'variant']).reset_index(drop=True)
epochs = pd.DataFrame(epoch_rows).sort_values(['variant', 'seed', 'epoch']).reset_index(drop=True)
missing_runs = pd.DataFrame(missing)
display(runs[['seed', 'model', 'best_epoch', 'best_val_ce', 'final_train_ce', 'minutes_per_epoch']].round(6))
if missing:
    print('Missing configurations (excluded from comparisons):')
    display(missing_runs[['variant', 'seed']])
export_tables.update(runs=runs, epoch_metrics=epochs, missing_runs=missing_runs)


## 2. Training and validation curves

Thin lines are individual training seeds; thick lines are means over available
seeds. Seed coverage is shown in each legend. CE uses the same smoothing (0.1).
Combined objective losses are not interchangeable across training variants.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for variant in LABELS:
    subset = epochs[epochs.variant == variant]
    if subset.empty:
        continue
    n = subset.seed.nunique()
    for ax, metric, title in zip(axes, ['ce', 'val_ce'], ['Training CE', 'Validation CE']):
        for _, group in subset.groupby('seed'):
            ax.plot(group.epoch, group[metric], color=COLORS[variant], alpha=0.22, linewidth=1)
        mean = subset.groupby('epoch')[metric].mean()
        ax.plot(mean.index, mean.values, color=COLORS[variant], linewidth=2.2,
                label=f'{LABELS[variant]} (n={n})')
        ax.set(xlabel='Epoch', ylabel='Smoothed CE', title=title,
               xticks=range(1, EPOCH_BUDGET + 1))
for ax in axes:
    ax.legend(fontsize=8)
fig.tight_layout()
figures['training_validation_curves'] = fig
plt.show()


In [ ]:
ncols = min(2, len(TRAINING_SEEDS))
nrows = int(np.ceil(len(TRAINING_SEEDS) / ncols))
fig, grid = plt.subplots(nrows, ncols, figsize=(13, 3.8 * nrows), squeeze=False)
for ax, seed in zip(grid.flat, TRAINING_SEEDS):
    for variant in LABELS:
        group = epochs[(epochs.seed == seed) & (epochs.variant == variant)]
        if group.empty:
            continue
        ax.plot(group.epoch, group.val_ce, color=COLORS[variant], label=LABELS[variant])
        selected = runs[(runs.seed == seed) & (runs.variant == variant)].iloc[0]
        ax.scatter(selected.best_epoch, selected.best_val_ce, color=COLORS[variant], s=40)
    ax.set(title=f'Training seed {seed}', xlabel='Epoch', ylabel='Validation CE')
    ax.legend(fontsize=8)
for ax in list(grid.flat)[len(TRAINING_SEEDS):]:
    ax.set_visible(False)
fig.suptitle('Validation curves and selected checkpoints', y=1.01)
fig.tight_layout()
figures['validation_by_training_seed'] = fig
plt.show()


## 3. Matched selected-checkpoint comparisons

Each candidate is compared with the **same-seed** M1. Negative changes favor the
candidate. These minima were used for checkpoint selection and can be optimistic;
the repeated paired evaluation below is a separate analysis of fixed checkpoints.
Sample SD describes training-seed variation, not a confidence interval.


In [ ]:
pair_rows, aggregate_rows = [], []
for variant in ('m2', 'feature', 'logit'):
    candidate = runs[runs.variant == variant].copy()
    baseline = runs[runs.variant == 'm1'][['seed', 'best_val_ce']].rename(columns={'best_val_ce': 'm1_val_ce'})
    matched = candidate.merge(baseline, on='seed', how='inner', validate='one_to_one')
    if matched.empty:
        continue
    matched['delta_val_ce'] = matched.best_val_ce - matched.m1_val_ce
    matched['relative_change_percent'] = 100 * matched.delta_val_ce / matched.m1_val_ce
    pair_rows.extend(matched[['seed', 'variant', 'best_epoch', 'best_val_ce', 'm1_val_ce',
                              'delta_val_ce', 'relative_change_percent']].to_dict('records'))
    aggregate_rows.append({'model': LABELS[variant], 'seeds': ', '.join(map(str, matched.seed)),
                           'n': len(matched), 'candidate_mean_best_val_ce': matched.best_val_ce.mean(),
                           'candidate_sample_sd': matched.best_val_ce.std(ddof=1),
                           'matched_m1_mean': matched.m1_val_ce.mean(),
                           'mean_delta_ce': matched.delta_val_ce.mean(),
                           'relative_mean_change_percent': 100 * (matched.best_val_ce.mean() / matched.m1_val_ce.mean() - 1),
                           'seeds_favoring_candidate': int((matched.delta_val_ce < 0).sum())})
selected_pairs = pd.DataFrame(pair_rows, columns=['seed', 'variant', 'best_epoch', 'best_val_ce',
    'm1_val_ce', 'delta_val_ce', 'relative_change_percent'])
selected_summary = pd.DataFrame(aggregate_rows)
display(selected_summary.round(6))
display(selected_pairs.round(6))
export_tables.update(selected_checkpoint_pairs=selected_pairs, selected_checkpoint_summary=selected_summary)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, variants, title in [(axes[0], ['feature'], 'Feature-only: selected-checkpoint change'),
                             (axes[1], ['m2', 'logit'], 'Combined and logit KD: selected-checkpoint change')]:
    for j, variant in enumerate(variants):
        group = selected_pairs[selected_pairs.variant == variant]
        offset = (j - (len(variants) - 1) / 2) * 0.18
        positions = [TRAINING_SEEDS.index(int(s)) + offset for s in group.seed]
        ax.scatter(positions, group.relative_change_percent, color=COLORS[variant], s=65,
                   label=f'{LABELS[variant]} (n={len(group)})')
    ax.axhline(0, color='#374151', linewidth=1)
    ax.set(xticks=range(len(TRAINING_SEEDS)), xticklabels=TRAINING_SEEDS,
           xlabel='Training seed', ylabel='Change from matched M1 (%)', title=title)
    ax.legend(fontsize=8)
fig.tight_layout()
figures['selected_checkpoint_relative_changes'] = fig
plt.show()


## 4. Training cost and auxiliary losses

Epoch times include training, validation and enabled diagnostics on the recorded
machine. They are **not inference latency** and do not establish deployment speed.
Auxiliary losses below are raw recorded terms; their weights differ by experiment.


In [ ]:
timing = runs.groupby(['variant', 'model']).agg(mean_minutes=('minutes_per_epoch', 'mean'),
             sample_sd_minutes=('minutes_per_epoch', 'std'), n=('seed', 'nunique')).reset_index()
display(timing.round(3))
fig, ax = plt.subplots(figsize=(9, 4))
positions = np.arange(len(timing))
ax.bar(positions, timing.mean_minutes, color=[COLORS[x] for x in timing.variant], width=0.6)
ax.set(xticks=positions, xticklabels=[f'{r.model}\n(n={r.n})' for r in timing.itertuples()],
       ylabel='Recorded minutes per epoch', title='Training + validation cost (not inference latency)')
for i, value in enumerate(timing.mean_minutes):
    ax.text(i, value + 0.2, f'{value:.2f}', ha='center', fontsize=9)
ax.set_ylim(0, timing.mean_minutes.max() * 1.18)
fig.tight_layout()
figures['recorded_epoch_time'] = fig
export_tables['training_time'] = timing
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for variant in ('m2', 'feature', 'logit'):
    subset = epochs[epochs.variant == variant]
    if subset.empty:
        continue
    for ax, metric, title in zip(axes, ['kd', 'tf'], ['Raw logit KD term', 'Raw feature term']):
        if not (subset[metric] != 0).any():
            continue
        mean = subset.groupby('epoch')[metric].mean()
        ax.plot(mean.index, mean.values, color=COLORS[variant], label=LABELS[variant])
        ax.set(title=title, xlabel='Epoch', ylabel='Mean raw training term')
for ax in axes:
    ax.legend(fontsize=8)
fig.tight_layout()
figures['auxiliary_loss_curves'] = fig
plt.show()


## 5. Repeated paired validation of fixed checkpoints

The latest compatible **full** evaluation is selected automatically unless
PAIRED_DIR is explicitly set. Smoke/partial evaluations are excluded. Every model
must have the same example/token/batch counts and the evaluated checkpoint hashes
must agree with the current run manifests. Statistics are recomputed from
trial_metrics.csv rather than trusting precomputed summaries.


In [ ]:
def compatible_evaluation(directory):
    path = directory / 'evaluation_manifest.json'
    if not path.is_file() or not (directory / 'trial_metrics.csv').is_file():
        return False
    m = json.loads(path.read_text())
    return (m.get('partial') is False and m.get('training_seeds') == TRAINING_SEEDS)

if PAIRED_DIR is None:
    choices = sorted([p for p in PAIRED_ROOT.iterdir() if p.is_dir() and compatible_evaluation(p)],
        key=lambda p: json.loads((p / 'evaluation_manifest.json').read_text()).get('created_utc', p.name)) if PAIRED_ROOT.is_dir() else []
    PAIRED_DIR = choices[-1] if choices else None
else:
    PAIRED_DIR = Path(PAIRED_DIR)
    if not PAIRED_DIR.is_absolute():
        PAIRED_DIR = REPO / PAIRED_DIR
    if not compatible_evaluation(PAIRED_DIR):
        raise ValueError('Selected evaluation is absent, partial, or has different training seeds.')

paired_trials = paired_deltas = paired_by_seed = pd.DataFrame()
paired_aggregate = {}
if PAIRED_DIR is None:
    print('No compatible full paired evaluation found. Remaining paired plots will be skipped.')
else:
    eval_manifest = json.loads((PAIRED_DIR / 'evaluation_manifest.json').read_text())
    if eval_manifest.get('snr_db') != 8.0 or eval_manifest.get('label_smoothing') != 0.1:
        raise ValueError('Unexpected paired-evaluation SNR or smoothing.')
    for entry in eval_manifest['inputs']:
        key = (entry['seed'], entry['label'])
        if key not in manifests:
            raise ValueError(f'Missing run manifest for evaluated checkpoint {key}')
        recorded = entry['recorded_manifest']['selected_checkpoint']['sha256']
        if recorded != manifests[key]['selected_checkpoint']['sha256']:
            raise ValueError(f'Checkpoint identity changed: {key}')
    paired_trials = pd.read_csv(PAIRED_DIR / 'trial_metrics.csv')
    metrics = ['batch_mean_ce', 'token_mean_ce', 'token_mean_nll']
    if not np.isfinite(paired_trials[metrics].to_numpy(dtype=float)).all():
        raise ValueError('Nonfinite paired metrics')
    if (paired_trials[['tokens', 'batches', 'examples']] <= 0).any().any():
        raise ValueError('Invalid paired counts')
    expected_entries = {(s, label) for s in TRAINING_SEEDS for label in ('m1', 'feature')}
    if {(e['seed'], e['label']) for e in eval_manifest['inputs']} != expected_entries:
        raise ValueError('Incomplete evaluated checkpoint identities')
    d, per_seed, paired_aggregate = paired_summary(paired_trials.to_dict('records'),
            eval_manifest['training_seeds'], eval_manifest['trial_seeds'])
    paired_deltas, paired_by_seed = pd.DataFrame(d), pd.DataFrame(per_seed)
    display(Markdown(f'**Paired input:** `{PAIRED_DIR.relative_to(REPO) if PAIRED_DIR.is_relative_to(REPO) else PAIRED_DIR}`'))
    display(paired_by_seed.round(6))
    display(pd.DataFrame(paired_aggregate).T.round(6))
    export_tables.update(paired_trial_metrics=paired_trials, paired_differences=paired_deltas,
                         paired_per_seed_summary=paired_by_seed)
    for filename in ('evaluation_manifest.json', 'trial_metrics.csv'):
        path = PAIRED_DIR / filename
        source_identities.append({'path': str(path), 'sha256': sha(path)})


In [ ]:
if not paired_deltas.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for ax, metric, title in zip(axes, ['delta_token_mean_ce', 'delta_token_mean_nll'],
                                 ['Token-weighted smoothed CE', 'Unsmoothed token NLL']):
        for x, seed in enumerate(TRAINING_SEEDS):
            values = paired_deltas.loc[paired_deltas.training_seed == seed, metric].to_numpy()
            jitter = np.linspace(-0.09, 0.09, len(values))
            ax.scatter(x + jitter, values, color='#64748b', s=28, alpha=0.8)
            ax.scatter(x, values.mean(), color='#059669' if values.mean() < 0 else '#dc2626',
                       marker='D', s=65, zorder=4)
        ax.axhline(0, color='#111827', linewidth=1)
        ax.set(xticks=range(len(TRAINING_SEEDS)), xticklabels=TRAINING_SEEDS,
               xlabel='Training seed', ylabel='Feature minus M1', title=title)
    fig.suptitle('Paired validation: dots = channel trials; diamonds = within-seed mean', fontsize=11)
    fig.tight_layout()
    figures['paired_differences_by_seed'] = fig
    plt.show()

    matrix = paired_deltas.pivot(index='training_seed', columns='trial_seed', values='delta_token_mean_ce')
    limit = max(float(np.abs(matrix.to_numpy()).max()), 1e-12)
    fig, ax = plt.subplots(figsize=(9, 4.3))
    heat = ax.imshow(matrix.to_numpy(), cmap='RdBu_r', vmin=-limit, vmax=limit, aspect='auto')
    ax.grid(False)
    ax.set(xticks=range(len(matrix.columns)), xticklabels=matrix.columns,
           yticks=range(len(matrix.index)), yticklabels=matrix.index,
           xlabel='Channel trial seed', ylabel='Training seed',
           title='Paired token-weighted CE difference (negative favors features)')
    for i in range(len(matrix.index)):
        for j in range(len(matrix.columns)):
            value = matrix.iloc[i, j]
            ax.text(j, i, f'{value:+.4f}', ha='center', va='center',
                    color='white' if abs(value) > 0.65 * limit else '#111827')
    fig.colorbar(heat, ax=ax, label='Feature minus M1')
    fig.tight_layout()
    figures['paired_ce_heatmap'] = fig
    plt.show()


## 6. Optional per-batch diagnostic plots

Set RUN_DIAGNOSTICS=True and list local raw CSV paths above. This can take time.
Temporary epoch summaries are computed using the streaming diagnostic tool.
Parameter norms are recorded **before** optimizer.step; clipping frequency alone
does not establish divergence. Partial epochs retain their actual batch counts.


In [ ]:
diagnostic_table = pd.DataFrame()
if RUN_DIAGNOSTICS:
    if not DIAGNOSTIC_PATHS:
        raise ValueError('List local diagnostic CSV paths in DIAGNOSTIC_PATHS first.')
    tables = []
    with tempfile.TemporaryDirectory() as tmp:
        for i, value in enumerate(DIAGNOSTIC_PATHS):
            path = Path(value)
            if not path.is_absolute():
                path = REPO / path
            destination = Path(tmp) / str(i)
            diagnostics(SimpleNamespace(input=path, output_dir=destination, clip=1.0))
            table = pd.read_csv(destination / 'diagnostics_by_epoch.csv')
            table['run'] = path.parent.name
            tables.append(table)
            source_identities.append({'path': str(path), 'sha256': sha(path)})
    diagnostic_table = pd.concat(tables, ignore_index=True)
    display(diagnostic_table.round(5))
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for run, group in diagnostic_table.groupby('run'):
        for ax, metric, title in zip(axes, ['gradient_mean', 'parameter_last_before_step', 'clipping_percent'],
                                     ['Mean gradient norm before clipping', 'Parameter norm before last update', 'Batches with gradient norm > 1 (%)']):
            ax.plot(group.epoch, group[metric], marker='o', label=run)
            ax.set(xlabel='Epoch', title=title)
    axes[0].legend(fontsize=6)
    fig.tight_layout()
    figures['batch_diagnostic_trends'] = fig
    export_tables['diagnostics_by_epoch'] = diagnostic_table
    plt.show()
else:
    print('Raw diagnostic analysis disabled; no large CSVs read.')


## 7. Export reproducible tables and figures

Exports go to a new timestamped folder. Figures are saved as PNG and PDF; tables
as CSV. Provenance includes input hashes, selected checkpoint identities, code
revision and analysis settings. Notebook outputs themselves should be cleared
before committing to Git. Raw data and checkpoints are not copied or modified.


In [ ]:
if SAVE_EXPORTS:
    output_dir = EXPORT_ROOT / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    output_dir.mkdir(parents=True, exist_ok=False)
    for name, table in export_tables.items():
        if not table.empty:
            table.to_csv(output_dir / f'{name}.csv', index=False)
    for name, fig in figures.items():
        fig.savefig(output_dir / f'{name}.png', bbox_inches='tight')
        fig.savefig(output_dir / f'{name}.pdf', bbox_inches='tight')
    provenance = {'created_utc': datetime.now(timezone.utc).isoformat(),
                  'git': git_state(REPO), 'training_seeds': TRAINING_SEEDS,
                  'epoch_budget': EPOCH_BUDGET, 'run_root': str(RUN_ROOT),
                  'paired_directory': str(PAIRED_DIR) if PAIRED_DIR else None,
                  'paired_statistics': paired_aggregate,
                  'inputs': source_identities,
                  'selected_checkpoints': runs[['seed', 'variant', 'checkpoint_sha256']].to_dict('records'),
                  'runtime': {'python': sys.version, 'pandas': pd.__version__,
                              'numpy': np.__version__, 'matplotlib': matplotlib.__version__},
                  'analysis_code_sha256': {
                      str(path.relative_to(REPO)): sha(path) for path in [
                          REPO / 'deepsc_reproduce/reproduce.py',
                          REPO / 'deepsc_reproduce/evaluate_paired_validation.py']},
                  'scope': 'Saved validation records and optional paired validation; no training or held-out test evaluation.'}
    (output_dir / 'analysis_manifest.json').write_text(json.dumps(provenance, indent=2, allow_nan=False))
    print('Saved tables and PNG/PDF graphs to:', output_dir)
else:
    print('Export disabled. Tables and figures remain in the notebook.')

if paired_aggregate:
    result = paired_aggregate['token_mean_ce']
    print(f"Paired token-weighted CE: mean feature-minus-M1 = {result['mean_paired_difference']:+.6f}; "
          f"sample SD across training seeds = {result['sample_sd_across_training_seeds']}; "
          f"features favored in {result['training_seeds_favoring_feature']}/{result['training_seed_count']} seeds.")
    print('Describe this as validation evidence. Do not infer statistical significance or decoded-sentence/test quality from these numbers alone.')
